In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Weight-and-Height 컬럼명 변경

import os
import glob
import pandas as pd

weight_raw_path = str(DATA_ROOT / 'AEGIS-Weight-and-Height/raw')
csv_files = glob.glob(os.path.join(weight_raw_path, '**/*.csv'), recursive = True)

# 기준 컬럼 비교용 세트 초기화
reference_columns = None
mismatch_files = []

# 파일별 컬럼 구조 확인
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str, nrows=0)  # 컬럼명만 로드
         # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())

        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

# 컬럼 불일치 파일 확인
if mismatch_files:
    print("🛑 컬럼 수/이름이 기준과 다른 파일 발견됨:")
    print(f"📌 기준 컬럼: {reference_columns}")
    for file, cols in mismatch_files:
        print(f"\n[🚫 불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼 목록: {cols}")
    sys.exit(1)

# 컬럼명 변경 매핑
column_map = {
    '환자번호#1': 'pt_id',
    '내원일시(최초1) #2': 'hos_dt_first',
    '진료일자#3': 'tx_date',
    '신체계측기록일시#4': 'body_ms_dt',
    '임상관찰기록종류코드#5': 'obx_cd',
    '임상관찰기록종류명#6': 'obx_nm',
    '외래VS측정일자#7': 'vs_date',
    '체중#8': 'weight',
    '신장#9': 'height',
    '복위#10': 'prone_bone',
    '흉위#11': 'chest',
    '두위#12': 'head',
    '앉은키#13': 'sit_height'
}

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype = str, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# basic_info 컬럼명 변경 (파일 저장)

import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
basic_info_path = str(DATA_ROOT / 'AEGIS-Basic-Info/raw')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(basic_info_path, '**/*.csv'), recursive=True)

# 2. 기준 컬럼 비교용 세트 초기화
reference_columns = None
mismatch_files = []

# 3. 파일별 컬럼 구조 확인
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str, nrows=0)  # 컬럼명만 로드
        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())

        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

# 4. 컬럼 불일치 파일 확인
if mismatch_files:
    print("🛑 컬럼 수/이름이 기준과 다른 파일 발견됨:")
    print(f"📌 기준 컬럼: {reference_columns}")
    for file, cols in mismatch_files:
        print(f"\n[🚫 불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼 목록: {cols}")
    sys.exit(1)

# 5. 컬럼명 매핑 딕셔너리
column_map = {
    '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '생년월일#3': 'birth_date',
    '성별#4': 'sex',
    '사망여부#5': 'death',
    '사망일자#6': 'death_date',
    '사망추정일자#7': 'death_est_date',
    '사망원인코드#8': 'death_cause_cd',
    '사망원인명#9': 'death_cause_nm',
    '사망원인출처코드#10': 'death_cd',
    '사망원인출처명#11': 'Death_nm',
    '국적코드#12': 'Region_cd',
    '국적명#13': 'Region_nm',
    '해외거주자여부#14': 'abroad',
    '우편번호#15': 'zip_code',
    '시도명#17': 'sido_nm',
    '시군구명#18': 'sigungu_nm',
    '개인정보동의여부#19': 'privacy',
    '민감정보동의여부#20': 'sensitive',
    '고유식별정보동의여부#21': 'unique',
    '주민번호성별#22': 'sex_cd'
}

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)


        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# nursing_initial_assessment 컬럼명 변경

import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
nursing_initial_path = str(DATA_ROOT / 'AEGIS-Nursing-Initial-Assessment/raw')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(nursing_initial_path, '**/*.csv'), recursive=True)

# 2. 기준 컬럼 비교용 세트 초기화
reference_columns = None
mismatch_files = []

# 3. 파일별 컬럼 구조 확인
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str, nrows=0)  # 컬럼명만 로드
        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())

        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

# 4. 컬럼 불일치 파일 확인
if mismatch_files:
    print("🛑 컬럼 수/이름이 기준과 다른 파일 발견됨:")
    print(f"📌 기준 컬럼: {reference_columns}")
    for file, cols in mismatch_files:
        print(f"\n[🚫 불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼 목록: {cols}")
    sys.exit(1)

# 5. 컬럼명 매핑 딕셔너리
nursing_column_map = {
     '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '예진도착시간#3': 'pre_tx_time',
    '응급실위치#4': 'ed_loc',
    '팀#5': 'team',
    '병상번호#6': 'bed_num',
    '주증상코드#7': 'cc_cd',
    '주증상명#8': 'cc_nm',
    '주증상내용#9': 'cc_text',
    '현병력#10': 'phx',
    '발병일시#11': 'os_dt',
    '발병시기#12': 'os_date',
    '발병시기단위코드#13': 'os_unit_cd',
    '발병시기단위#14': 'os_unit',
    'V/S_SBP#15': 'sbp',
    'V/S_DBP#16': 'dbp',
    'V/S_PR#17': 'pr',
    'V/S_RR#18': 'rr',
    'V/S_temp코드#19': 'temp_cd',
    'V/S_temp구분#20': 'temp_cat',
    'V/S_temp내용#21': 'temp',
    'V/S_SpO2코드#22': 'spo2_cd',
    'V/S_SpO2코드명#23': 'spo2_cdnm',
    'V/S_SpO2값#24': 'spo2_value',
    'V/S_SpO2(O2)_l/min#25': 'spo2',
    'V/S_SpO2방법코드#26': 'spo2_mtcd',
    'V/S_SpO2방법#27': 'spo2_mt',
    'V/S_BWT#28': 'bwt',
    'V/S측정불가#29': 'vs_missing',
    '여행력여부#30': 'travel',
    '여행국가#31': 'travel_contry',
    '기침여부#32': 'couph',
    '가래여부#33': 'phlegm',
    '콧물여부#34': 'runny_nose',
    '특기사항#35': 'etc_text',
    '최초KTAS등록일시#36': 'f_ktas_dt',
    '최초KTAS코드#37': 'f_ktas_cd',
    '최초KTAS#38': 'f_ktas',
    '최초KTAS레벨#39': 'f_ktas_lv',
    '최종KTAS등록일시#42': 'l_ktas_dt',
    '최종KTAS코드#43': 'l_ktas_cd',
    '최종KTAS#44': 'l_ktas_lv',
    '최종KTAS레벨#45': 'l_ktas',
    '최초감염항목코드#48': 'infection_cd_1st',
    '최초감염항목#49': 'infection_type',
    '감염항목코드#50': 'infection_cd',
    '감염항목명#51': 'infection_nm',
    '환자반응코드#52': 'pt_response_cd',
    '환자반응#53': 'pt_response',
    '응급의료관리코드#54': 'ed_manage_cd',
    '응급의료관리#55': 'ed_manage',
    '통증여부#56': 'pain',
    '통증부위#57': 'pain_site',
    '통증척도#58': 'pain_scale',
    '통증점수#59': 'pain_score',
    '통증양상#60': 'pain_type',
    '통증양상etc#61': 'pain_type_text',
    '통증빈도#62': 'pain_freq',
    '통증빈도etc#63': 'pain_freq_text',
    '통증기간#64': 'pain_duration',
    '과거력값-고혈압#65': 'phx_hypertension',
    '과거력값-당뇨#66': 'phx_diabetes',
    '과거력값-결핵#67': 'phx_tuberculosis',
    '과거력값-간염#68': 'phx_hepatitis',
    '과거력값-etc#69': 'phx_etc',
    '과거력특기사항#70': 'phx_text',
    '과거력-갑상선#71': 'phx_thyroid',
    '과거력-알레르기#72': 'phx_allergy',
    '과거력-Heart Disease#73': 'phx_heart',
    '과거력유무#74': 'phx',
    '과거력고혈압#75': 'phx_hypertension2',
    '과거력당뇨#76': 'phx_diabetes2',
    '과거력결핵#77': 'phx_tuberculosis2',
    '과거력간염#78': 'phx_hepatitis2',
    '과거력etc#79': 'phx_etc2',
    '과거력etc내용#80': 'phx_text2',
    '과거력심장질환#81': 'phx_heart2',
    '과거력갑상선#82': 'phx_thyroid2',
    '과거력알러지특기#83': 'phx_allergy2',
    '수술력#84': 'surgical_history',
    '보호자확인#86': 'protector',
    '보관물품#87': 'stuff',
    '질병분류코드#91': 'disease_cd',
    '질병분류#92': 'disesase',
    '최초질병분류코드#93': 'f_disease_cd',
    '최초질병분류#94': 'f_disease',
    '내원수단코드#95': 'hos_mtcd',
    '내원수단#96': 'hos_mt',
    '내원경로코드#97': 'hos_route_cd',
    '내원경로#98': 'hos_route',
    '외래코드#99': 'opd_cd',
    '외래명#100': 'opd_nm',
    '검사실코드#101': 'lab_cd',
    '검사실#102': 'lab',
    '검사실직접입력#103': 'lab_text',
    '전원_타병원#104': 'transfer_another',
    '외상의도구분코드#105': 'trauma_cd',
    '외상의도구분#106': 'trauma',
    '외상손상기전코드#107': 'trauma_injury_cd',
    '외상손상기전#108': 'trauma_injury',
    '외상손상기전교통사고코드#109': 'trauma_ta_cd',
    '외상손상기전교통사고#110': 'trauma_ta',
    '교통사고당사자코드#111': 'ta_participant_cd',
    '교통사고당사자#112': 'ta_participant',
    '안전벨트#113': 'seatbelt',
    '아동용좌석#114': 'child_seat',
    '전면에어백#115': 'front_airbag',
    '측면에어백#116': 'side_airbag',
    '헬맷#117': 'helmet',
    '무릎 및 관절보호대#118': 'protective_gear',
    '구명조끼#119': 'life_jacket',
    '착용안함#120': 'not_wearing',
    '비해당#121': 'ta_na',
    '미상#122': 'ta_unknown',
    '진료과코드#123': 'depart_cd',
    '진료과#124': 'depart',
    '입원결정일#127': 'ad_date',
    '상병명#128': 'dig_nm',
    '퇴실일시#133': 'dc_dt',
    '응급진료결과코드#134': 'ed_result_cd',
    '응급진료결과#135': 'ed_result',
    '응급진료병실호수#136': 'ed_ward',
    '응급진료취소사유코드#137': 'ed_cancle_cd',
    '응급진료취소사유#138': 'ed_cancle_cause',
    '취소etc사유#139': 'cancle_cause',
    '입원구분코드#140': 'ad_cd',
    '입원구분#141': 'ad',
    '입원병동코드#142': 'ad_ward_cd',
    '입원병동#143': 'ad_ward',
    '입원경로코드#144': 'ad_route_cd',
    '입원경로#145': 'ad_route',
    '귀가구분코드#146': 'dc_cd',
    '귀가구분#147': 'dc',
    '전원구분코드#148': 'transfer_cd',
    '전원구분#149': 'transfer',
    '전원병원내용#150': 'transfer_text',
    '사망구분코드#151': 'death_cd',
    '사망구분#152': 'death',
    '퇴원시점의식상태코드#157': 'dc_avpu_cd',
    '퇴원시점의식상태#158': 'dc_avpu',
    '환자상태구분코드#159': 'pt_condition_cd',
    '환자상태구분#160': 'pt_condition',
    '이송장소코드#161': 'transport_place_cd',
    '이송장소#162': 'transport_place',
    '이송장소사유#163': 'transport_place_cause',
    '이송방법코드#164': 'transport_mtcd',
    '이송방법#165': 'transport_mt',
    '응급이송수단코드#166': 'ed_transport_cd',
    '응급이송수단#167': 'ed_transport',
    '이송수단사유#168': 'transport_cause',
    '분만장주증상코드#169': 'dr_cc_cd',
    '분만장주증상#170': 'dr_cc',
    '분만장주증상내용#171': 'dr_cc_text',
    'LMP#172': 'lmp',
    'EDC#173': 'edc',
    'CDEC#174': 'dced',
    '재태기간(주)#175': 'gestation_week',
    '재태기간(일)#176': 'gestation_day',
    '양막파수유무#177': 'amniotomy',
    '양막파수일시#178': 'amniotomy_dt',
    '이슬유무#179': 'show',
    '이슬일시#180': 'show_dt',
    '출산력F수#181': 'parity_f',
    '출산력P수#182': 'parity_p',
    '출산력A1수#183': 'parity_a1',
    '출산력A2수#184': 'parity_a2',
    '출산력L1수#185': 'parity_l1',
    '출산력L2수#186': 'parity_l2',
    '이전분만형태#187': 'pre_delivery',
    '월경규칙적여부#188': 'menstrual_regular',
    '월경주기#189': 'menstrual_cycle',
    '월경기간(from)#190': 'menstrual_from',
    '월경기간(to)#191': 'menstrual_to',
    'FHT1#192': 'fht1',
    'FHT2#193': 'fht2',
    'FHT3#194': 'fht3',
    'FHT4#195': 'fht4',
    'Cx(cm)#196': 'cx_cm',
    'Cx(%)#197': 'cx_percent',
    '기형아검사여부#198': 'fetal_anomaly',
    '기형아검사#199': 'fetal_anomaly_result',
    '기형아검사내용#200': 'fetal_anomaly_text',
    '중기SONO여부#201': 'mid_sono',
    '중기SONO#202': 'mid_sono_result',
    '중기SONO내용#203': 'mid_sono_text',
    '50g_GTT여부#204': 'gtt50',
    '50g_GTT#205': 'gtt50_result',
    '50g_GTT내용#206': 'gtt50_text',
    '100g_GTT여부#207': 'gtt100',
    '100g_GTT#208': 'gtt100_result',
    '100g_GTT내용#209': 'gtt100_text',
    'Amniocentesis여부#210': 'amniocentesis',
    'Amniocentesis#211': 'amniocentesis_result',
    'Amniocentesis내용#212': 'amniocentesis_text',
    '문진_혈액형#213': 'mh_blood_type',
    '문진_RPR#214': 'mh_rpr',
    '문진_HBsAg#215': 'mh_hmsag',
    '문진_HIV#216': 'mh_hiv',
    '검사결과일1#219': 'ex1_date',
    '검사결과1_혈액형#220': 'ex1_blood_type',
    '검사결과1_RPR#221': 'ex1_rpr',
    '검사결과1_HBsAg#222': 'ex1_hbsag',
    '검사결과1_HIV#223': 'ex1_hiv',
    '검사결과1_확인자ID#224': 'ex1_person_id',
    '검사결과1_확인자#225': 'ex1_person',
    '검사결과일2#226': 'ex2_date',
    '검사결과2_혈액형#227': 'ex2_blood_type',
    '검사결과2_RPR#228': 'ex2_rpr',
    '검사결과2_HBsAg#229': 'ex2_hbsag',
    '검사결과2_HIV#230': 'ex2_hiv',
    '검사결과2_확인자ID#231': 'ex2_person_id',
    '검사결과2_확인자#232': 'ex2_person',
    '합병증커멘트내용#233': 'complication_text',
    'NEDIS구분코드#234': 'nedis_cd',
    'NEDIS구분#235': 'nedis',
    'NEDIS확인일시#236': 'nedis_dt',
    'NEDIS응급실송신여부#237': 'nedis_ed',
    'NEDIS입원송신여부#238': 'nedis_ad',
    '전입병원종류내용#239': 'trans_in_text',
    '전입병원종류내용유형명#240': 'trans_in_nm',
    '전입병원코드#241': 'trans_in_cd',
    '전입병원#242': 'trans_in',
    '전출병원종류내용#243': 'trans_out_text',
    '전출병원종류내용유형명#244': 'trans_out_nm',
    '전출병원요양기관번호#245': 'nursing_home_num',
    '전출병원요양기관#246': 'nursing_home',
    '전출병원코드#247': 'trans_out_cd',
    '전출병원#248': 'trans_out',
    '입원후퇴실시전출병원종류내용#249': 'dc_trans_out_text',
    '입원후퇴실시전출병원종류내용#250': 'dc_trans_out_text2',
    '입원후퇴실시전출병원코드#251': 'dc_trans_out_cd',
    '입원후퇴실시전출병원명#252': 'dc_trans_out_nm',
    '입원장발급일자#253': 'ad_certification_date',
    '퇴원후건강계획호출#254': 'dc_health_plan'
}

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=nursing_column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# order_civ1 컬럼명 변경

import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
order_civ1_path = str(DATA_ROOT / 'AEGIS-Order-CIV1/raw')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(order_civ1_path, '**/*.csv'), recursive=True)

# 2. 기준 컬럼 비교용 세트 초기화
reference_columns = None
mismatch_files = []

# 3. 파일별 컬럼 구조 확인
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str, nrows=0)  # 컬럼명만 로드

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())

        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

# 4. 컬럼 불일치 파일 확인
if mismatch_files:
    print("🛑 컬럼 수/이름이 기준과 다른 파일 발견됨:")
    print(f"📌 기준 컬럼: {reference_columns}")
    for file, cols in mismatch_files:
        print(f"\n[🚫 불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼 목록: {cols}")
    sys.exit(1)

# 5. 컬럼명 매핑 딕셔너리
civ1_column_map = {
    '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '구분코드#3': 'cat_cd',
    '구분#4': 'cat',
    '투약시작일시#5': 'med_start_dt',
    '투약종료일시#6': 'med_end_dt',
    '투약상태코드#7': 'med_cd',
    '투약상태#8': 'med_status',
    '투여용법#9': 'adm_type',
    'DC여부#10': 'dc',
    '기록구분코드#11': 'record_cd',
    '기록구분#12': 'record',
    '계획구분코드#13': 'plan_cat_cd',
    '계획구분#14': 'plan_cd',
    'CIV투여용량#15': 'civ_adm_dosage',
    'CIV투여용량단위#16': 'civ_adm_unit',
    '계산기준코드#17': 'cal_cd',
    '계산기준#18': 'cal_standard',
    '주입속도용량#19': 'infusion_rate_dosage',
    '주입속도단위#20': 'infusion_rate_unit',
    '주입속도시간단위#21': 'infusion_rate_tm_unit',
    '주입용량#22': 'infusion_dosage',
    '주입용량단위#23': 'infusion_dosage_unit',
    '주입용량시간단위#24': 'infusion_dosage_tm_unit',
    '투약용도체중체크여부#25': 'med_weight_check',
    '투약용도체중값#26': 'med_weight_value',
    '처방일자#28': 'order_date',
    '처방코드#29': 'order_cd',
    '처방명(입력)#30': 'order_nm_input',
    '처방명(마스터)#31': 'order_nm_master',
    '추가처방여부#32': 'add_order',
    '처방투여용량#33': 'order_amd',
    '처방투여용량단위#34': 'order_amd_unit',
    '처방용액여부#35': 'order_solution',
    '처방용질여부#36': 'order_solute',
    '당일투여계획횟수#37': 'adm_times_plan',
    '당일최대투여횟수#38': 'adm_times_max',
    '당일투여간격시간분#39': 'adm_time_interval',
    '일당투여계획횟수#40': 'adm_times_day',
    '투약상세기록일시#41': 'adm_dt',
    '수술일자#43': 'surgery_date'
}

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=civ1_column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)


        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = "utf-8")
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# civ2 컬럼명 변경

import os
import glob
import pandas as pd
import sys

# 1. 대상 경로
order_civ2_path = str(DATA_ROOT / 'AEGIS-Order-CIV2/raw')  # 경로 수정 필요
csv_files = glob.glob(os.path.join(order_civ2_path, '**/*.csv'), recursive=True)

# 3. 기준 컬럼 구조 확인
reference_columns = None
mismatch_files = []

# 3. 파일별 컬럼 구조 확인
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str, nrows=0)  # 컬럼명만 로드
        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())

        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

# 4. 컬럼 불일치 파일 확인
if mismatch_files:
    print("🛑 컬럼 수/이름이 기준과 다른 파일 발견됨:")
    print(f"📌 기준 컬럼: {reference_columns}")
    for file, cols in mismatch_files:
        print(f"\n[🚫 불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼 목록: {cols}")
    sys.exit(1)

# 2. 컬럼명 매핑 딕셔너리
column_map = {
    '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '처방일자#3': 'order_date',
    '처방코드#4': 'order_cd',
    '처방명#5': 'order_nm',
    '내원구분코드#6': 'hos_cat_cd',
    '내원구분명#7': 'hos_cat_nm',
    'CIV입력일시#8': 'civ_dt',
    'CIV계산구분코드#9': 'civ_cal_cd',
    'CIV계산구분코드명#10': 'civ_cal_nm',
    'CIV주입속도값#11': 'civ_rate',
    'CIV주입속도단위#12': 'civ_rate_unit',
    'CIV주입속도시간단위#13': 'civ_rate_time_unit',
    'CIV주입용량#14': 'civ_infusion_dosage',
    'CIV주입용량단위#15': 'civ_dosage_unit',
    'CIV주입용량시간단위#16': 'civ_dosage_time_unit',
    'CIV투약용체중값#17': 'civ_med_weight_value',
    '처방확인일시(간호)#18': 'order_nursing_dt',
    'CIV투약상태코드#19': 'civ_med_cd',
    'CIV투약상태코드명#20': 'civ_med_cdnm',
    '추가처방구분코드#21': 'ad_med_cv',
    '추가처방구분코드명#22': 'ad_med_cdnm',
    'DC구분코드#23': 'dc_cd',
    'DC구분코드명#24': 'dc_dcnm',
    'CIV일련번호#25': 'civ_num',
    '처방발생원코드#26': 'order_occur_cd',
    '처방발생원코드명#27': 'order_occur_cdnm',
    '연계처방코드#28': 'linkage_order_cd',
    '연계처방명#29': 'linkage_order_nm',
    '약처방일자#30': 'medicine_order_date',
    '약처방sno#31': 'medicine_order_sno'
}


# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# snomed 컬럼명 변경

import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
dx_snomed_path = str(DATA_ROOT / 'AEGIS-Dx-SNOMED/raw')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(dx_snomed_path, '**/*.csv'), recursive=True)

# 2. 컬럼명 매핑 딕셔너리
column_map = {
    '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '진료일시#3': 'tx_dt',
    '환자임상진단일련번호#4': 'pt_dig_num',
    '내원구분코드#5': 'hos_cat_cd',
    '내원구분#6': 'hos_cat',
    '진료과코드#7': 'depart_cd',
    '진료과#8': 'depart',
    '임상진단코드#9': 'dig_cd',
    '임상진단명#10': 'dig_nm',
    '임상진단약어#11': 'dig_abbreviation',
    '임상진단메모#12': 'dig_memo',
    '주임상진단여부#13': 'cdig',
    'RULE_Out여부#14': 'rule_out',
    '확정상병여부#15': 'confirm_dig',
    '환자상병특이내용#16': 'pt_dig_memo',
    '진단부위코드#17': 'dig_site_cd',
    '진단부위#18': 'dig_site',
    '입원진료과코드#23': 'ad_depart_cd',
    '입원진료과명#24': 'ad_depart_nm',
    '퇴원진료과코드#25': 'dc_depart_cd',
    '퇴원진료과명#26': 'dc_depart_nm',
    'SNOMED_CONP_ID#27': 'snomed_conp_id',
    'SNOMED_CONP_NAME#28': 'snomed_conp_nm',
    'SNOMED_DSCR_ID#29': 'snomed_dscr_id',
    'SNOMED_DSCR_NAME#30': 'snomed_dscr_nm'
}

# 3. 기준 컬럼 구조 비교
reference_columns = None
mismatch_files = []

for file in csv_files:
    try:
        table_df = pd.read_csv(file)
        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())
        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

if mismatch_files:
    print("🛑 컬럼 구조 불일치 파일 발견:")
    print(f"기준 컬럼: {reference_columns}")
    for file, cols in mismatch_files:
        print(f"\n[불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼명: {cols}")
    sys.exit(1)

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# kcd 컬럼명 변경

import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
dx_kcd_path = str(DATA_ROOT / 'AEGIS-Dx-KCD/raw')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(dx_kcd_path, '**/*.csv'), recursive=True)

# 2. 컬럼명 매핑
column_map = {
    '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '진료일시#3': 'tx_dt',
    '내원구분코드#4': 'visit_cd',
    '내원구분#5': 'visit_cat',
    '진료과코드#6': 'depart_cd',
    '진료과#7': 'depart',
    '상병코드#8': 'dig_cd',
    '상병명#9': 'dig_nm',
    '주상병여부#10': 'cdig',
    '전염병여부#11': 'infection',
    '입원진료과코드#12': 'ad_depart_cd',
    '입원진료과명#13': 'ad_depart_nm',
    '퇴원진료과코드#14': 'dc_depart_cd',
    '퇴원진료과명#15': 'dc_depart_nm',
    '상병영문#20': 'dig_eng',
    '상병한글#21': 'dig_kor',
    '재원상병구분코드#22': 'ad_dig_cd',
    '재원상병구분#23': 'ad_dig_cat',
    'ICD대분류코드#24': 'icd_large_cd',
    'ICD대분류명#25': 'icd_large_nm',
    'ICD중분류코드#26': 'icd_middle_cd',
    'ICD중분류명#27': 'icd_middle_nm',
    'ICD소분류코드#28': 'icd_small_cd',
    'ICD소분류명#29': 'icd_small_nm',
    '상병그룹코드#30': 'dia_group_cd',
    '의무기록POA구분코드#31': 'poa_cd',
    '의무기록POA구분#32': 'poa_cat'
}

# 3. 기준 컬럼 확인
reference_columns = None
mismatch_files = []

for file in csv_files:
    try:
        table_df = pd.read_csv(file)
        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())
        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

# 4. 불일치 파일 출력 후 중단
if mismatch_files:
    print("🛑 컬럼 구조 불일치 파일 발견:")
    print(f"기준 컬럼: {reference_columns}")
    for file, cols in mismatch_files:
        print(f"\n[불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼명: {cols}")
    sys.exit(1)

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# flowsheet 컬럼명 변경

import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
flowsheet_path = str(DATA_ROOT / 'AEGIS-Flowsheet-Value/raw2')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(flowsheet_path, '**/*.csv'), recursive=True)

# 2. 컬럼명 매핑 (위에서 정의한 column_map 사용)
column_map = {
    '환자번호#1': 'pt_id',
    '내원일시(최초1) #2': 'hos_dt_first',
    '임상관찰기록종류코드#3': 'obs_cd',
    '임상관찰기록종류#4': 'obs_cat',
    'Tab코드#5': 'tab_cd',
    'Tab명#6': 'tab_nm',
    '항목그룹#7': 'cat_group',
    '항목그룹명#8': 'cat_group_nm',
    '항목#9': 'cat',
    '항목명#10': 'cat_nm',
    '항목부위명#11': 'cat_site_nm',
    'Value_ID#12': 'value_id',
    'Value명#13': 'value_nm',
    '항목기록일시#14': 'cat_dt',
    '항목기록값#15': 'cat_value',
    '항목기타내용#16': 'cat_text',
    'Value점수#17': 'value_score'
}

# # 3. 기준 컬럼 구조 검증
# reference_columns = None
# mismatch_files = []


# for file in csv_files:
#     print(file)
#     try:
#         table_df = pd.read_csv(file, dtype = str, sep=",", quotechar ='"', lineterminator = '\n')

#         # Unnamed 0 컬럼 제거
#         if 'Unnamed: 0' in df.columns:
#             df.drop(columns = ['Unnamed: 0'], inplace = True)

#         columns = tuple(df.columns.tolist())
#         if reference_columns is None:
#             reference_columns = columns
#         elif columns != reference_columns:
#             mismatch_files.append((file, columns))
#     except Exception as e:
#         print(f"[❗로드 오류] {file} → {e}")

# # 4. 불일치 파일 중단
# if mismatch_files:
#     print("🛑 컬럼 구조가 다른 파일이 존재합니다:")
#     for file, cols in mismatch_files:
#         print(f"\n[불일치] {file}")
#         print(f"컬럼 수: {len(cols)}")
#         print(f"컬럼명: {cols}")
#     sys.exit(1)

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype = str, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# administer
import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
administer_path = str(DATA_ROOT / 'AEGIS-Administer-Med/raw')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(administer_path, '**/*.csv'), recursive=True)

column_map = {
    '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '처방일자#3': 'order_date',
    '입원일시#4': 'ad_dt',
    '진료과코드#5': 'depart_cd',
    '진료과명#6': 'depart_nm',
    '처방코드#7': 'order_cd',
    '처방명#8': 'order_nm',
    '액팅구분코드#10': 'act_cd',
    '액팅구분명#11': 'act_nm',
    '병동약주사액팅회차#12': 'act_times',
    '액팅예약일자#13': 'act_reserv_date',
    '액팅예정일시#14': 'act_schedue_dt',
    '액팅실시일시#15': 'act_dt',
    '액팅수정일시#16': 'act_edit_dt',
    '액팅종류코드#19': 'act_cat_cd',
    '액팅종류#20': 'act_cat',
    '액팅입력경로코드#22': 'act_mt_cd',
    '액팅입력경로#23': 'act_mt',
    '병동코드#25': 'ward_cd',
    '병동명#26': 'ward_nm',
    '처방일련번호#27': 'order_num'
}


# 2. 기준 컬럼 구조 검증
reference_columns = None
mismatch_files = []

for file in csv_files:
    try:
        table_df = pd.read_csv(file)
        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())
        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

if mismatch_files:
    print("🛑 컬럼 구조 불일치 파일 존재:")
    for file, cols in mismatch_files:
        print(f"\n[불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼명: {cols}")
    sys.exit(1)

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding= 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# basic order 컬럼 변경

import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
basic_order_path = str(DATA_ROOT / 'AEGIS-Basic-Order/raw')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(basic_order_path, '**/*.csv'), recursive=True)

column_map = {
    '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '내원구분코드#3': 'visit_cd',
    '내원구분명#4': 'visit_nm',
    '진료과코드#5': 'depart_cd',
    '진료과명#6': 'depart_nm',
    '처방일자#7': 'order_date',
    '처방코드#8': 'order_cd',
    '처방명#9': 'order_nm',
    '임시처방여부#11': 'temporal_order',
    'DC구분코드#12': 'dc_cd',
    'DC구분#13': 'dc',
    '입력사용자구분코드#22': 'user_cd',
    '입력사용자구분#23': 'user',
    '진료센터코드#24': 'tx_center_cd',
    '진료센터명#25': 'tx_center_nm',
    '처방발생원코드#26': 'order_sr_cd',
    '처방발생원#27': 'order_sr',
    '병동코드#28': 'ward_cd',
    '병동명#29': 'ward_nm',
    '처방형태코드#30': 'order_form_cd',
    '처방형태명#31': 'order_form_nm',
    '처방분류코드#32': 'order_cat_cd',
    '처방분류명#33': 'order_cat_nm',
    '처방종류코드#34': 'order_type_cd',
    '처방종류명#35': 'order_type_nm',
    '처방종류상세코드#36': 'order_type_text_cd',
    '처방종류상세#37': 'orter_type_text',
    'SLIP코드#38': 'slip_cd',
    'SLIP명#39': 'slip_nm',
    '정규처방완료일시#40': 'order_finish_dt',
    '처방확인일시#41': 'order_confirm_dt',
    '진료일시#42': 'tx_dt',
    '인증구분코드#43': 'certification_cd',
    '인증구분#44': 'certification',
    '인증일시#47': 'certification_dt',
    '처방생성구분#52': 'order_generate_cat',
    '최초등록일시#53': 'first_dt'
}


# 2. 기준 컬럼 구조 검증
reference_columns = None
mismatch_files = []

for file in csv_files:
    try:
        table_df = pd.read_csv(file)
        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())
        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

if mismatch_files:
    print("🛑 컬럼 구조 불일치 파일 존재:")
    for file, cols in mismatch_files:
        print(f"\n[불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼명: {cols}")
    sys.exit(1)

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
# order med


import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
order_med_path = str(DATA_ROOT / 'AEGIS-Order-Med1/raw')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(order_med_path, '**/*.csv'), recursive=True)

column_map = {
    '환자번호#1': 'pt_id',
    '내원일시#2': 'hos_dt',
    '횟수#4': 'visit_times',
    '처방일자#5': 'order_date',
    '처방등록일시#6': 'order_regi_dt',
    '진료(입원)일시#7': 'tx_dt',
    '약묶음번호#8': 'medication_num',
    '처방코드#9': 'order_cd',
    '처방명#10': 'order_nm',
    '처방일련번호#11': 'order_num',
    '내원구분코드#12': 'hos_cd',
    '내원구분#13': 'hos_cat',
    '총용량#14': 'sum_capacity',
    '총용량단위#15': 'sum_capacity_unit',
    '횟수#16': 'times',
    '일수#17': 'days',
    '선택단위구분코드#18': 'select_unit_cd',
    '선택단위구분#19': 'select_unit',
    '포장단위코드#20': 'pack_unit_cd',
    '포장단위#21': 'pack_unit',
    '포장단위투여량#22': 'pack_unit_capacity',
    '함량단위코드#23': 'content_unit_cd',
    '함량단위#24': 'content_unit',
    '함량단위투여용량#25': 'content_adm_capacity',
    '투약Kg당용량#26': 'med_kg_capacity',
    '용법코드#27': 'usage_cd',
    '용법#28': 'usage',
    '성분내용#29': 'ingredient',
    '처방형태코드#30': 'order_form_cd',
    '처방형태#31': 'order_form',
    '처방분류코드#32': 'order_classify_cd',
    '처방분류#33': 'order_classify',
    '간호상태코드#34': 'nursing_cd',
    '간호상태#35': 'nursing',
    'DC구분코드#36': 'dc_cd',
    'DC구분#37': 'dc',
    'DC처방일시#38': 'dc_order_dt',
    '처방종류코드#39': 'order_type1_cd',
    '처방종류#40': 'order_type1',
    '처방종류상세코드#41': 'order_type2_cd',
    '처방종류상세#42': 'order_type2',
    '약적용구분코드#43': 'medicine_apply_cd',
    '약적용구분#44': 'medicine_apply',
    '처방확인일시#45': 'order_check_dt',
    'ATC코드#46': 'atc_cd',
    'ATC코드명#47': 'atc_cdnm',
    'ATC레벨1코드#48': 'atc1_cd'
}


# 2. 기준 컬럼 구조 검증
reference_columns = None
mismatch_files = []

for file in csv_files:
    try:
        table_df = pd.read_csv(file)
        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)
        columns = tuple(table_df.columns.tolist())
        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

if mismatch_files:
    print("🛑 컬럼 구조 불일치 파일 존재:")
    for file, cols in mismatch_files:
        print(f"\n[불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼명: {cols}")
    sys.exit(1)

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)



        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")


In [ ]:
#  order test 처리


import os
import glob
import pandas as pd
import sys

# 1. 경로 설정
order_test_path = str(DATA_ROOT / 'AEGIS-Order-Test/raw2')  # 실제 경로로 수정
csv_files = glob.glob(os.path.join(order_test_path, '**/*.csv'), recursive=True)

column_map = {
    '환자번호#1': 'pt_id',
    #'내원일시#2': 'hos_dt',
    '처방코드#3': 'order_cd',
    '처방명#4': 'order_nm',
    '처방일자#5': 'order_date',
    '접수일시#6': 'register_dt',
    '시행일시#7': 'conduct_dt',
    '판독일시#8': 'reading_dt',
    '검사결과보고일시#9': 'test_result_report_dt',
    '보고일시#10': 'report_dt',
    '진료일시#11': 'tx_dt',
    '검체코드1#12': 'specimen1_cd',
    '검체명1#13': 'specimen1_nm',
    '병리구분코드#14': 'pathology_cd',
    '병리구분명#15': 'pathology_nm',
    'Body Part(영상)#16': 'body_part',
    'Modality(영상)#17': 'modality',
    '검사코드#18': 'test_cd',
    '검사명#19': 'test_nm',
    '검사진행상태코드#20': 'test_ing_cd',
    '검사결과(text)#21': 'test_result_text',
    '결론및진단#22': 'test_dig',
    'Gross결과#23': 'gross',
    '검사결과수치값#24': 'test_result_value',
    '검사결과단위명#25': 'test_result_unit',
    '검사결과표준값#26': 'test_result_standard',
    '검사결과특정내용#27': 'test_result_specific',
    'SLIP코드#28': 'slip_cd',
    'SLIP명#29': 'slip_nm'
}


# 2. 기준 컬럼 구조 검증
reference_columns = None
mismatch_files = []

for file in csv_files:
    try:
        table_df = pd.read_csv(file)

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        columns = tuple(table_df.columns.tolist())
        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            mismatch_files.append((file, columns))
    except Exception as e:
        print(f"[❗로드 오류] {file} → {e}")

if mismatch_files:
    print("🛑 컬럼 구조 불일치 파일 존재:")
    for file, cols in mismatch_files:
        print(f"\n[불일치] {file}")
        print(f"컬럼 수: {len(cols)}")
        print(f"컬럼명: {cols}")
    sys.exit(1)

# 6. 컬럼명 변경 및 저장
saved_file_count = 0

for file in csv_files:
    try:
        table_df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')

        # Unnamed 0 컬럼 제거
        if 'Unnamed: 0' in table_df.columns:
            table_df.drop(columns = ['Unnamed: 0'], inplace = True)

        standardized_df = table_df.rename(columns=column_map)

        # 컬럼 수 검증
        if len(table_df.columns) != len(standardized_df.columns):
            print(f"[🛑 중단] 컬럼 수 변경 발생: {file}")
            print(f"변경 전: {len(table_df.columns)}, 변경 후: {len(standardized_df.columns)}")
            sys.exit(1)

        # 행 수 비교 추가
        if len(table_df) != len(standardized_df):
            print(f"[🛑 중단] 행 수 변경 발생: {file}")
            print(f"변경 전 행 수: {len(table_df)}, 변경 후 행 수: {len(standardized_df)}")
            sys.exit(1)


        # 하위 폴더를 유지해 처리본 경로를 생성한다.
        save_path = deidentified_path(file)
        os.makedirs(os.path.dirname(save_path), exist_ok = True)

        standardized_df.to_csv(save_path, index = False, encoding = 'utf-8')
        saved_file_count += 1
        print(f"저장 완료: {save_path}")

    except Exception as e:
        print(f"저장 오류: {file} -> {e}")

print("\n 파일 처리:")
print(f"원본 파일 수 : {len(csv_files)}")
print(f"저장된 파일 수 : {saved_file_count}")
